In [ ]:
import pandas as pd
import numpy as np

# =============================
# 1. LOAD ALL CSV FILES
# =============================

train_smote       = pd.read_csv("training_cv_table_smote_5.csv")
train_weighted    = pd.read_csv("training_cv_table_weighted_5.csv")
train_pca_smote   = pd.read_csv("training_cv_table_pca_smote.csv")
train_pca_weighed = pd.read_csv("training_cv_table_pca_weighted.csv")

test_smote        = pd.read_csv("test_table_smote_5.csv")
test_weighted     = pd.read_csv("test_table_weighted_5.csv")
test_pca_smote    = pd.read_csv("test_table_pca_smote.csv")
test_pca_weighted = pd.read_csv("test_table_pca_weighted.csv")

# Add Dataset / Type
for df_part in [train_smote, train_weighted, train_pca_smote, train_pca_weighed]:
    df_part["Dataset"] = "Train"

for df_part in [test_smote, test_weighted, test_pca_smote, test_pca_weighted]:
    df_part["Dataset"] = "Test"

train_smote["Type"]       = "SMOTE"
train_pca_smote["Type"]   = "SMOTE"
test_smote["Type"]        = "SMOTE"
test_pca_smote["Type"]    = "SMOTE"

train_weighted["Type"]       = "Weighted"
train_pca_weighed["Type"]    = "Weighted"
test_weighted["Type"]        = "Weighted"
test_pca_weighted["Type"]    = "Weighted"

# Tag source: Raw vs PCA
for df_raw in [train_smote, train_weighted, test_smote, test_weighted]:
    df_raw["Source"] = "Raw"

for df_pca in [train_pca_smote, train_pca_weighed, test_pca_smote, test_pca_weighted]:
    df_pca["Source"] = "PCA"


In [ ]:
df = pd.concat(
    [
        train_smote,
        train_weighted,
        test_smote,
        test_weighted,
        train_pca_smote,
        test_pca_smote,
        train_pca_weighed,
        test_pca_weighted,
    ],
    ignore_index=True
)
df["SourceType"] = df["Source"] + " / " + df["Type"]
df["Features"] = df["Features"].astype(str).str.strip()
df["n_features"] = df["Features"].apply(lambda x: len(x.split(",")))




In [ ]:
import re

def normalize_model_name(x):
    x = x.lower()

    if "knn" in x:
        return "KNN"
    if "logistic" in x:
        return "Logistic Regression"
    if "decision tree" in x:
        return "Decision Tree"
    if "random forest" in x:
        return "Random Forest"
    if "xgboost" in x or "xgb" in x:
        return "XGBoost"

    return "OTHER"

df["Model_base"] = df["Model"].apply(normalize_model_name)


In [ ]:
print(df.columns)


In [ ]:
df_base = df[df["n_features"] == 2].copy()
df_multi = df[df["n_features"] > 2].copy()


In [ ]:
summary_pairs = df_base.pivot_table(
    index="Features",
    columns=["Dataset", "Model_base", "Type"],
    values=["Precision", "Recall", "F1-Score", "ROC-AUC"],
    aggfunc="mean"
)

summary_pairs


In [ ]:
rows = []

for (model, dataset, mtype), group in df_base.groupby(["Model_base", "Dataset", "Type"]):

    best_f1 = group["F1-Score"].max()

    for _, r in group.iterrows():
        rows.append({
            "Model": model,
            "Dataset": dataset,   # Train / Test
            "Type": mtype,        # SMOTE / Weighted
            "Features": r["Features"],
            "F1": r["F1-Score"],
            "ROC-AUC": r["ROC-AUC"],
            "ΔF1_vs_best": r["F1-Score"] - best_f1  # <= 0, best pair = 0
        })

pair_effects_df = pd.DataFrame(rows)
pair_effects_df


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

model_to_plot = "XGBoost"

subset = pair_effects_df[
    (pair_effects_df["Model"] == model_to_plot) &
    (pair_effects_df["Dataset"] == "Test")
].sort_values("F1", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=subset, x="Features", y="F1", hue="Type")
plt.xticks(rotation=45, ha="right")
plt.title(f"Test F1 for all 2-feature combinations – {model_to_plot}")
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a consistent, accessible palette
palette = sns.color_palette("Set1")  # fixed mapping for clarity
hue_order = ["Raw / Weighted","Raw / SMOTE", "PCA / Weighted","PCA / SMOTE"]  # enforce consistent order

keep_models = ["KNN", "Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"]

# initial filter
df_pairs = df[(df["Dataset"] == "Test") & (df["Model_base"].isin(keep_models))]

# guard: drop combinations where any row has n_features != 2
valid_combos = (
    df_pairs.groupby("Combination #")["n_features"]
    .apply(lambda x: (x == 2).all())
)
df_pairs = df_pairs[df_pairs["Combination #"].isin(valid_combos[valid_combos].index)]

for model_name in df_pairs["Model_base"].unique():
    sub = df_pairs[df_pairs["Model_base"] == model_name].copy()
    sub = sub.sort_values("F1-Score", ascending=False)

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(
        data=sub,
        x="Combination #",
        y="F1-Score",
        hue="SourceType",
        palette=palette,
        hue_order=hue_order,
        dodge=True
    )

    # X-axis: rotate labels for readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

    # Legend: move outside, clear title
    ax.legend(title="Source Type", bbox_to_anchor=(1.05, 1), loc="upper left")

    # Titles and labels
    ax.set_title(f"Test F1 – 2-feature combinations, Raw vs PCA ({model_name})")
    ax.set_xlabel("Feature Combination ID")
    ax.set_ylabel("Test F1-Score")

    plt.tight_layout()
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a consistent, colorblind-safe palette
palette = sns.color_palette("Set1")  # or try "colorblind"
hue_order = ["Raw / Weighted","Raw / SMOTE", "PCA / Weighted","PCA / SMOTE"]  # enforce consistent order

keep_models = ["KNN", "Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"]

df_pairs = df[(df["Dataset"] == "Test") & (df["Model_base"].isin(keep_models))]
valid_combos = (
    df_pairs.groupby("Combination #")["n_features"]
    .apply(lambda x: (x == 2).all())
)
df_pairs = df_pairs[df_pairs["Combination #"].isin(valid_combos[valid_combos].index)]
combos_to_exclude = df_pairs["Combination #"].unique()

df_multi = df[df["Model_base"].isin(keep_models)].copy()
df_multi = df_multi[~df_multi.index.isin(df_pairs.index)]
df_multi = df_multi[~df_multi["Combination #"].isin(combos_to_exclude)]

for model_name in df_multi["Model_base"].unique():
    sub = df_multi[df_multi["Model_base"] == model_name].copy()
    sub = sub.sort_values("F1-Score", ascending=False)

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(
        data=sub,
        x="Combination #",
        y="F1-Score",
        hue="SourceType",
        palette=palette,
        hue_order=hue_order,
        dodge=True
    )

    # Improve X-axis: show only unique combos, rotate for readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

    # Legend: move outside, title clearer
    ax.legend(title="Source Type", bbox_to_anchor=(1.05, 1), loc="upper left")

    # Titles and labels
    ax.set_title(f"Test F1 Multiple Feature Combination Raw vs PCA ({model_name})")
    ax.set_xlabel("Feature Combination ID")
    ax.set_ylabel("Test F1-Score")

    plt.tight_layout()
    plt.show()

In [ ]:
keep_models = ["KNN", "Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"]

for model_name in keep_models:

    sub_pairs = df_pairs[
        (df_pairs["Model_base"] == model_name) &
        (df_pairs["Dataset"] == "Test")
    ]

    sub_multi = df_multi[
        (df_multi["Model_base"] == model_name) &
        (df_multi["Dataset"] == "Test")
    ]

    plt.figure(figsize=(14, 5))

    # 2-feature combinations
    plt.subplot(1, 2, 1)
    sns.barplot(
        data=sub_pairs.sort_values("F1-Score", ascending=False),
        x="Combination #",
        y="F1-Score",
        hue="Type"
    )
    plt.xticks(rotation=0)
    plt.title(f"{model_name} – 2-feature combos (Test F1)")
    plt.xlabel("Combination #")
    plt.ylabel("Test F1-Score")

    # multi-feature combinations
    plt.subplot(1, 2, 2)
    sns.barplot(
        data=sub_multi.sort_values("F1-Score", ascending=False),
        x="Combination #",
        y="F1-Score",
        hue="Type"
    )
    plt.xticks(rotation=0)
    plt.title(f"{model_name} – Multi-feature combos (Test F1)")
    plt.xlabel("Combination #")
    plt.ylabel("Test F1-Score")

    plt.tight_layout()
    plt.show()



In [ ]:
effects = []

for model in df["Model_base"].unique():

    base_model = df_base[df_base["Model_base"] == model]

    for _, r in df_multi[df_multi["Model_base"] == model].iterrows():

        # Use first feature as baseline anchor
        first_feat = r["Features"].split(",")[0].strip()

        base_row_train = base_model[
            (base_model["Dataset"] == "Train")
            & (base_model["Features"].str.contains(first_feat))
        ]

        base_row_test = base_model[
            (base_model["Dataset"] == "Test")
            & (base_model["Features"].str.contains(first_feat))
        ]

        if base_row_train.empty or base_row_test.empty:
            continue

        b_train = base_row_train.iloc[0]
        b_test = base_row_test.iloc[0]

        effects.append(
            {
                "Model": model,
                "Combination": r["Features"],
                "Baseline": b_train["Features"],
                # TRAIN EFFECTS
                "Train_ΔPrecision": r["Precision"] - b_train["Precision"],
                "Train_ΔRecall": r["Recall"] - b_train["Recall"],
                "Train_ΔF1": r["F1-Score"] - b_train["F1-Score"],
                "Train_ΔAUC": r["ROC-AUC"] - b_train["ROC-AUC"],
                # TEST EFFECTS
                "Test_ΔPrecision": r["Precision"] - b_test["Precision"],
                "Test_ΔRecall": r["Recall"] - b_test["Recall"],
                "Test_ΔF1": r["F1-Score"] - b_test["F1-Score"],
                "Test_ΔAUC": r["ROC-AUC"] - b_test["ROC-AUC"],
                # META
                "Model_Type": r["Type"],  # SMOTE or Weighted
                "n_features": r["n_features"],
            }
        )


effects_df = pd.DataFrame(effects)
effects_df

In [ ]:
pivot_train_f1 = effects_df.pivot_table(
    index="Combination",
    columns=["Model", "Model_Type"],
    values="Train_ΔF1",
    aggfunc="mean"
)
pivot_train_f1


In [ ]:
pivot_test_f1 = effects_df.pivot_table(
    index="Combination",
    columns=["Model", "Model_Type"],
    values="Test_ΔF1",
    aggfunc="mean"
)
pivot_test_f1


In [ ]:
pivot_train_auc = effects_df.pivot_table(
    index="Combination",
    columns=["Model", "Model_Type"],
    values="Train_ΔAUC",
    aggfunc="mean"
)
pivot_train_auc


In [ ]:
pivot_test_auc = effects_df.pivot_table(
    index="Combination",
    columns=["Model", "Model_Type"],
    values="Test_ΔAUC",
    aggfunc="mean"
)
pivot_test_auc


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", font_scale=1.1)


In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(
    pivot_train_f1,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=.5,
    cbar_kws={'label': 'Train ΔF1'}
)
plt.title("Effect of Feature Combinations on Training F1 (ΔF1)")
plt.xlabel("Model and Training Type")
plt.ylabel("Feature Combination")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(
    pivot_test_f1,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=.5,
    cbar_kws={'label': 'Test ΔF1'}
)
plt.title("Effect of Feature Combinations on Test F1 (ΔF1)")
plt.xlabel("Model and Training Type")
plt.ylabel("Feature Combination")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(
    pivot_train_auc,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    linewidths=.5,
    cbar_kws={'label': 'Train ΔAUC'}
)
plt.title("Effect of Feature Combinations on Training AUC (ΔAUC)")
plt.xlabel("Model and Training Type")
plt.ylabel("Feature Combination")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(
    pivot_test_auc,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    linewidths=.5,
    cbar_kws={'label': 'Test ΔAUC'}
)
plt.title("Effect of Feature Combinations on Test AUC (ΔAUC)")
plt.xlabel("Model and Training Type")
plt.ylabel("Feature Combination")
plt.tight_layout()
plt.show()
